![NWM](img/NWM.png)

# Use HydroData to Retrieve Modeled and Observed Snow Data for a Watershed of Interest  
Authors: Irene Garousi-Nejad (igarousi@cuahsi.org), Danielle Tijerina-Kreuzer (dtijerina@cuahsi.org)  
Last updated: January 2026  

This notebook utilized the Princeton HydroData repository to gather [SWE point observations](https://hf-hydrodata.readthedocs.io/en/latest/available_datasets.html#point-observations) and [ParFlow CONUS1 outputs](https://hf-hydrodata.readthedocs.io/en/latest/available_datasets.html#parflow-simulation-outputs).  
See the `hf_hydrodata` [Getting Started](https://hf-hydrodata.readthedocs.io/en/latest/getting_started.html) page for more information on the package and account. 


## 1. Setup

### 1a. Python Environment  

Ensure that the `nwm_env` conda environment is selected as your Jupyter kernel. This environment should already be created if you followed the instructions under section "Creating your HydroLearnEnv Virtual Environment" in the `getting_started.md` file.

Import the libraries needed to run this notebook:

In [ ]:
import os

prefix = os.environ['CONDA_PREFIX']
os.environ['PROJ_LIB'] = os.path.join(prefix, 'share', 'proj')

import sys
import pyproj
import pandas as pd
import numpy as np
import xarray as xr
import geopandas as gpd
from dask.distributed import Client
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import hf_hydrodata as hf
import hvplot.xarray

from utils import nwm_utils

%load_ext autoreload
%autoreload 2


### 1b. Register Pin and Access HydroData

To access the HydroData catalog you will need to sign up for a [HydroFrame account](https://hydrogen.princeton.edu/signup) (do this only once), [create a 4-digit PIN](https://hydrogen.princeton.edu/pin), and register your pin in order to have access to the HydroData datasets (you will do this in the next code cell below). To note, you PIN will expire after 7 days and will need to recreate it after that time. 

In [ ]:
# You need to register on https://hydrogen.princeton.edu/pin 
# and run the following with your registered information
# before you can use the hydrodata utilities
hf.register_api_pin("dtt2@princeton.edu", "7837")

### 1c. Dask  

We'll use dask to parallelize our code. To manage parallel computation and visualize progress of long-running tasks, we initialize a Dask “cluster,” which defines how many workers are used and how much computing power each worker has. 

In this setup, we create a Dask client with `Client(n_workers=6, threads_per_worker=1, memory_limit='2GB')`, which launches a cluster with 6 workers. Each worker uses a single thread, typically mapped to one CPU core, allowing for efficient parallel processing across 6 cores. Each worker also has a memory limit of 2 GB, for a total of up to 12 GB across the cluster.


In [ ]:
# use a try accept loop so we only instantiate the client
# if it doesn't already exist.
try:
    print('Dashboard link:', client.dashboard_link)
except:    
    # The client should be customized to your workstation resources.
    client = Client(n_workers=6, threads_per_worker=1, memory_limit='2GB') 
    print('Dashboard link:', client.dashboard_link)
print(client)

## 2. Set Paths

In [ ]:
# Start and end times of a water year (to note, these dates were chosen to align with the PFCONUS1 early 2000s runs)
StartDate = '2003-10-01'
EndDate = '2005-09-30'

# Path to save results (obs and mod stands for observation and modeled, respectively)
OBS_OutputFolder = './obs_outputs_PF' 
MOD_OutputFolder = './mod_outputs_PF'

## 3. Retrieve Observed Snow Data 

### 3a. Define the watershed of interest

One of the simplest ways to gather data and model output from HydroData is by specifying a [Hydrologic Unit Code](https://www.usgs.gov/national-hydrography/watershed-boundary-dataset). Before we retrieve any hydrologic information, we need to indicate a HUC8 code and use it to gather snow water equivalent (SWE) observations from SNOTEL sites  

✏️ If you have a specific HUC8 in mind, you can change the variable `huc_8_code` below.

In [ ]:
# ✏️ Specify HUC8 ID and Name for watershed of interest
huc_8_code = '14020001'  # East-Taylor HUC-8
print(f'HUC-8 ID: {huc_8_code}')

huc_8_name = 'East-Taylor'
print(f'HUC-8 name: {huc_8_name}')

### 3b. Explore the available SWE data in a watershed  

<div style="color:black;background-color:#f5f5f5; padding:10px; border-left: 5px solid #007acc;">
<h4>📖 Did you know?</h4>
<p>The Snow Telemetry (SNOTEL) network, managed by the USDA Natural Resources Conservation Service (NRCS), monitors snowpack conditions across key watersheds in the western United States to support water supply forecasting and climate monitoring. SNOTEL sites are fully automated stations that continuously measure snow water equivalent (SWE), snow depth, precipitation, temperature, and other meteorological variables throughout the year. Unlike manual snow survey programs, SNOTEL provides high-temporal-resolution observations that enable near–real-time assessment of snowpack evolution and interannual variability. These data are widely used for operational forecasting, drought assessment, and long-term climate analysis. </p>
</div>

Explore what SWE data is available at sites within the HUC ID you specified that operated during WY2004 and WY2005. If you want to check other variables besides SWE, you can change the `variable` argument name. 

In [ ]:
avail_df = hf.get_site_variables(variable = "swe",
                        huc_id = [huc_8_code], grid = 'conus1',
                        date_start = StartDate, date_end = EndDate)

# View first five records
avail_df.head(5)

### 3c. Map the SNOTEL stations inside the HUC-08 watershed that have available data in the selected time range  
To note here, we are using pre-loaded shape files for the East-Taylor HUC8, which are located in the `/domain_data/` directory.

In [ ]:
### Select station locations that fall within the HUC8 watershed

# Path to the watershed shapefile that was just created
watershed = 'domain_data/East-Taylor_14020001.shp'
watershed_gdf = gpd.read_file(watershed).to_crs(epsg=4326)

# Create GeoDataFrame of all available stations
filtered_all_stations_gdf = gpd.GeoDataFrame(
    avail_df,
    geometry=gpd.points_from_xy(
        avail_df.longitude,
        avail_df.latitude
    ),
    crs="EPSG:4326"
)
print("Sites CRS:", filtered_all_stations_gdf.crs)

# Combine watershed polygons into one geometry
watershed_union = watershed_gdf.geometry.unary_union

# Filter stations that fall within the watershed
sites_in_watershed = filtered_all_stations_gdf[
    filtered_all_stations_gdf.geometry.within(watershed_union)
].copy()

sites_in_watershed.reset_index(drop=True, inplace=True)

print(f"Total sites in watershed: {len(sites_in_watershed)}")




Plot these sites on a map. Then, hover over the pins to see the site names.

In [ ]:
## TODO: REPLACE WITH CSSI_EVALUATION.PLOTS FUNCTIONS

# this may take a moment to load, but it should pop up in a new window
m = nwm_utils.plot_sites_within_domain(sites_in_watershed, watershed_gdf, zoom_start=9)
m

## 4. Retrieve SNOTEL point observations and metadata from HydroData  
Use the `hf.get_point_data()` function to retrieve daily, start-of-day SWE from SNOTEL sites:

In [ ]:
# Create a folder to save observations
isExist = os.path.exists(OBS_OutputFolder)
if isExist == True:
    exit
else:
    os.mkdir(OBS_OutputFolder)

### 4a. Get HydroData Observed SWE
Gather the SNOTEL data for all stations within the watershed and save CSV:

In [ ]:
# Request point observations data
data_df = hf.get_point_data(dataset="snotel", variable="swe", temporal_resolution="daily", aggregation="sod",
                         date_start=StartDate, date_end=EndDate,
                         huc_id=[huc_8_code], grid='conus1')
                         #polygon=watershed_bbox, polygon_crs=watershed_crs)

# save
data_df.to_csv(f'./{OBS_OutputFolder}/df_{huc_8_name}_{huc_8_code}_SNOTEL.csv', index=False)

# Ensure date column is datetime
data_df["date"] = pd.to_datetime(data_df["date"])

# View first five records
data_df.head(5)

### 4b. Get Metadata for HydroData Observed SWE
Also, retrieve the metadata for the same stations we retrieved SWE observations for:

In [ ]:
# Request site-level attributes for these sites
metadata_df = hf.get_point_metadata(dataset="snotel", variable="swe", temporal_resolution="daily", aggregation="sod",
                                 date_start=StartDate, date_end=EndDate,
                                 huc_id=['14020001'], grid='conus1')

# save
metadata_df.to_csv(f'./{OBS_OutputFolder}/df_{huc_8_name}_{huc_8_code}_SNOTEL_metadata.csv', index=False)

# View first five records
metadata_df.head(5)

The metadata file is an important addition to the observations and it is recommended to always gather and save this for the observations you are using (particularly to support reproducibility within an open-science workflow). The saved file has useful attributes like site names, first and last date of available data, lat/lon, and the query URL.  

Additionally, the metadata contains **ParFlow-CONUS1 and ParFlow-CONUS2 `i,j` indices, which indicate the exact model domain grid cell the observation aligns with**. This is a useful HydroData feature that removes the need for users to manually match station latitude/longitude coordinates to the appropriate model grid cell, as this spatial mapping is handled directly within HydroData. We will use these indices below to extract PF-CONUS1 modeled SWE for each SNOTEL station in the section below. 

## 5. Retrieve ParFlow-CONUS1 Modeled Snow Data

In [ ]:
# Create a folder to save results
isExist = os.path.exists(MOD_OutputFolder)
if isExist == True:
    exit
else:
    os.mkdir(MOD_OutputFolder)

The following section retrieves ParFlow-CONUS1 data for each SNOTEL site within our HUC-08 watershed. The code identifies the CONUS1 `i,j` indices associated with each SNOTEL site, indicated in the `metadata_df`. It then extracts the CONUS1 modeled SWE output for the site and the period of interest, returning the result as a DataFrame. To fairly compare with SNOTEL, which reports SWE once daily at the start of the local day, model output is aggregated by day, using the argment `"temporal_resolution": "daily"`. Finally, the processed data is saved as a CSV file for each site.  

### 5a. ParFlow CONUS1 Model Dataset Information
We can print some information about the model output dataset by using the `hf.get_catalog_entry()` to get the CONUS1 model dataset metadata. 

In [ ]:
conus1_options = {
    "dataset": "conus1_baseline_mod",
    "variable": "swe"
}
hf.get_catalog_entry(conus1_options)

Before we gather model outputs at the specific SNOTEL sites, we can visualize SWE across our HUC-08. This is plotted for one day at 1km lateral resolution.

In [ ]:
# retrieve gridded PF-CONUS1 SWE for the entire HUC8 watershed
grid_swe_options = {
        "dataset": "conus1_baseline_mod",
        "variable": "swe",
        "temporal_resolution": "daily",
        "start_time": '2004-04-01', ### TO NOTE: the gridded function has exclusive end date, so this is hardcoded for now 
        "end_time": '2004-04-02',
        "huc_id": huc_8_code
    }
    
    # Get gridded data
grid_swe = hf.get_gridded_data(grid_swe_options)

In [ ]:
grid_swe_map = xr.DataArray(grid_swe[0], dims=("y", "x"), name="SWE")
grid_swe_map.hvplot.image(cmap="YlGnBu", colorbar=True, aspect="equal", title=f"{huc_8_name} Gridded SWE on 2004-04-01")


Now, grab the PF-CONUS1 modeled SWE from the SNOTEL site locations. Here we use the CONUS1 i and j indices from the `metadata_df` and grab the SWE from those grid cells.  

In [ ]:
# Copy data_df to model_df so we have the same timestamps and site_id structure
model_df = data_df.copy()

# Set all non-date columns to NaN to prepare for filling in model data
non_date_cols = model_df.columns.difference(["date"])
model_df[non_date_cols] = np.nan

# Rename site_id columns for PF outputs 
model_df.columns = [
    col if col == "date" else col.replace(":SNTL", "") + ":PFCONUS1"
    for col in model_df.columns
]

Use the function `hf.get_gridded_data()` and PF-CONUS1 `i,j` indices to select the SWE output for the correct location and time period: 

In [ ]:
# Loop over each station in metadata_df
for idx, row in metadata_df.iterrows():
    site_id = row["site_id"]  # original SNTL site_id
    col_name = site_id.replace(":SNTL", "") + ":PFCONUS1"  # corresponding column in model_df
    conus_i = int(row["conus1_i"])
    conus_j = int(row["conus1_j"])
    
    # Build options dict for this station
    options = {
        "dataset": "conus1_baseline_mod",
        "variable": "swe",
        "temporal_resolution": "daily",
        "start_time": '2003-10-01', ### TO NOTE: the gridded function has exclusive end date, so this is hardcoded for now 
        "end_time": '2005-10-01',
        "grid_point": [conus_i, conus_j]
    }
    
    # Get gridded data
    data = hf.get_gridded_data(options)
    
    # Fill column in model_df
    # Convert to numeric in case hf returns lists or other types
    model_df[col_name] = np.squeeze(np.array(data))

# Ensure date column is datetime
model_df["date"] = pd.to_datetime(model_df["date"])

# Save
model_df.to_csv(f'./{MOD_OutputFolder}/df_{huc_8_name}_{huc_8_code}_PFCONUS1.csv', index=False)
    
model_df.head(5)

## 6. Quick plot sanity check  
Plot a simple timeseries of modeled and observed SWE to make sure our data retrieval was successful. 

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

ax.plot(data_df["date"], model_df["380:CO:PFCONUS1"], label="Modeled", linewidth=2)

ax.plot(data_df["date"], data_df["380:CO:SNTL"], label="Observed", linewidth=2)

ax.set_xlabel("Date")
ax.set_ylabel("SWE (mm)")

# Date formatting for x-axis
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%Y'))

ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)

plt.tight_layout()